# ⚡ Day 02a: UDP — Fast, Connectionless Delivery

---

## 🎯 What You'll Master Today
- UDP header (just 8 bytes!)
- Connectionless, no reliability
- TCP vs UDP comparison
- When to use UDP

---

### 🎭 The Analogy

**TCP** = registered mail (tracked, signed, confirmed delivery).  
**UDP** = throwing postcards out a window (fast, no confirmation, some may be lost).

```
╔══════════════════════════════════════════════════════════════════════╗
║              UDP HEADER (8 bytes — that's it!)                      ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  ┌──────────────────┬──────────────────┐                            ║
║  │  Source Port (2B) │  Dest Port (2B)  │                            ║
║  ├──────────────────┼──────────────────┤                            ║
║  │  Length (2B)      │  Checksum (2B)   │                            ║
║  └──────────────────┴──────────────────┘                            ║
║  │              Payload                  │                           ║
║  └───────────────────────────────────────┘                           ║
║                                                                      ║
║  Compare: TCP header = 20+ bytes, UDP header = 8 bytes             ║
║  No seq numbers, no ACK, no handshake, no flow control             ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. TCP vs UDP Side-by-Side
# ============================================

comparison = [
    ("Connection",     "Connection-oriented (handshake)",   "Connectionless (fire and forget)"),
    ("Reliability",    "Guaranteed delivery (ACK/retx)",    "No guarantee (best effort)"),
    ("Ordering",       "In-order delivery (seq numbers)",   "No ordering guarantee"),
    ("Flow control",   "Sliding window",                    "None"),
    ("Congestion ctrl","Slow start, AIMD",                  "None (can flood network)"),
    ("Header size",    "20-60 bytes",                       "8 bytes"),
    ("Speed",          "Slower (overhead)",                  "Faster (minimal overhead)"),
    ("Streaming",      "Byte stream",                       "Message-oriented (datagrams)"),
    ("Broadcast",      "No",                                 "Yes (multicast/broadcast)"),
    ("Error checking", "Checksum + retransmission",         "Checksum only (optional in IPv4)"),
]

print("  ╔═══════════════╦══════════════════════════╦═══════════════════════════════╗")
print("  ║ Feature        ║ TCP                      ║ UDP                           ║")
print("  ╠═══════════════╬══════════════════════════╬═══════════════════════════════╣")
for feature, tcp, udp in comparison:
    print(f"  ║ {feature:13} ║ {tcp:24} ║ {udp:29} ║")
print("  ╚═══════════════╩══════════════════════════╩═══════════════════════════════╝")

In [ ]:
# ============================================
# 2. UDP vs TCP Overhead Demo
# ============================================

def calculate_overhead(protocol, data_size, n_packets):
    ip_header = 20
    
    if protocol == "TCP":
        transport_header = 20
        # Handshake: 3 packets, termination: 4 packets
        handshake_bytes = 3 * (ip_header + transport_header)
        teardown_bytes = 4 * (ip_header + transport_header)
        # ACKs for each data packet
        ack_bytes = n_packets * (ip_header + transport_header)
        total_overhead = (n_packets * transport_header) + handshake_bytes + teardown_bytes + ack_bytes
    else:  # UDP
        transport_header = 8
        handshake_bytes = 0
        teardown_bytes = 0
        ack_bytes = 0
        total_overhead = n_packets * transport_header
    
    total_data = n_packets * data_size
    total_sent = total_data + total_overhead
    
    return {
        "protocol": protocol,
        "data": total_data,
        "overhead": total_overhead,
        "total": total_sent,
        "efficiency": total_data / total_sent * 100,
    }

print("  ═══ Overhead Comparison: 100 packets of 100 bytes each ═══\n")

for proto in ["TCP", "UDP"]:
    r = calculate_overhead(proto, 100, 100)
    print(f"  {r['protocol']}:")
    print(f"    Data sent:    {r['data']:,} bytes")
    print(f"    Overhead:     {r['overhead']:,} bytes")
    print(f"    Total:        {r['total']:,} bytes")
    print(f"    Efficiency:   {r['efficiency']:.1f}%\n")

print("  For small frequent messages (gaming, DNS), UDP wins big!")
print("  For large transfers (files), TCP's overhead is negligible.")

In [ ]:
# ============================================
# 3. When to Use UDP
# ============================================

print("""
  ═══ WHEN TO USE UDP ═══

  ┌────────────────────────────────────────────────────────────────┐
  │ Use Case              │ Why UDP?                              │
  ├───────────────────────┼───────────────────────────────────────┤
  │ DNS queries           │ Small, single request/response        │
  │ Video streaming       │ Tolerate loss, need low latency       │
  │ Online gaming         │ Real-time, stale data useless         │
  │ VoIP (voice calls)    │ Delay worse than losing a sample      │
  │ IoT sensors           │ Lightweight, frequent small messages  │
  │ DHCP                  │ No IP yet, can't establish TCP conn   │
  │ SNMP                  │ Simple network mgmt, small messages   │
  │ NTP                   │ Time sync, tiny packets               │
  │ Multicast/Broadcast   │ TCP doesn't support 1-to-many        │
  └───────────────────────┴───────────────────────────────────────┘

  Rule of thumb:
  - Need reliability? → TCP
  - Need speed + tolerate loss? → UDP
  - Need custom reliability? → UDP + app-layer protocol (QUIC, RTP)

  Modern trend: build reliability on TOP of UDP
    - QUIC (HTTP/3): reliable, congestion-controlled, encrypted
    - RTP/RTCP: real-time transport for audio/video
    - WebRTC: peer-to-peer audio/video (uses UDP underneath)
""")

In [ ]:
# ============================================
# 4. DNS over UDP vs TCP
# ============================================

print("""
  ═══ DNS: UDP vs TCP ═══

  DNS uses BOTH:

  UDP (port 53) — default for queries:
    Client: "What's the IP of google.com?" (1 packet)
    Server: "142.250.80.46" (1 packet)
    Total: 2 packets. Done!

  TCP (port 53) — used when:
    1. Response > 512 bytes (overflow, many records)
    2. Zone transfers (AXFR) between DNS servers
    3. DNS over TLS (DoT) on port 853

  With TCP, same query would be:
    SYN → SYN-ACK → ACK → Query → Response → FIN → ACK → FIN → ACK
    Total: 9 packets. 4.5x more overhead!

  This is why DNS defaults to UDP.
""")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | TCP vs UDP? | TCP: reliable, ordered, connection. UDP: fast, unreliable, connectionless |
| 2 | UDP header size? | 8 bytes: src_port(2) + dst_port(2) + length(2) + checksum(2) |
| 3 | When would you use UDP? | DNS, streaming, gaming, VoIP — when speed > reliability |
| 4 | Can you build reliable UDP? | Yes — QUIC (HTTP/3) adds reliability, encryption on top of UDP |
| 5 | Why DNS uses UDP? | Small requests, single packet. TCP would need 9 packets (handshake overhead) |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • UDP: 8-byte header, no handshake, no reliability      │
## │  • Perfect for: DNS, streaming, gaming, VoIP, IoT       │
## │  • QUIC = reliable protocol built on UDP (HTTP/3)        │
## │  • UDP supports broadcast/multicast (TCP cannot)         │
## │  • Trade-off: speed vs reliability                       │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Flow & Congestion Control** — How TCP avoids flooding the network